# EDA — Turbofan Engine Degradation (C-MAPSS schema)

Exploratory analysis behind the feature-selection and modeling choices in `src/features.py` and `src/train.py`.

**Note on the data:** this sandbox had no internet access, so `data/raw/` contains synthetic
data generated by `data/generate_cmapss_like_data.py` matching the real NASA C-MAPSS schema
exactly (same 26 columns, same degradation-curve shape). Swap in the real dataset from
https://www.kaggle.com/datasets/behrad3d/nasa-cmaps and every cell below still works unmodified.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '../src')
from features import load_raw, load_rul, add_rul_labels_train, COLUMNS, SENSOR_COLS

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (11, 4)

train = load_raw('../data/raw/train_FD001.txt')
train = add_rul_labels_train(train)
print(train.shape)
train.head()

## 1. How many engines, how long do they run?

In [ ]:
life_per_unit = train.groupby('unit_number')['time_in_cycles'].max()
print(life_per_unit.describe())

plt.figure()
sns.histplot(life_per_unit, bins=25, color='#4fa8a0')
plt.xlabel('Cycles to failure')
plt.title('Distribution of run-to-failure length across the fleet')
plt.show()

## 2. Sensor trends across engine life

The key modeling question: which of the 21 sensors actually carry a
degradation signal, and which are flat / noise? We plot each sensor's
mean trajectory (as % of life) across all engines.

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(18, 14), sharex=True)
axes = axes.flatten()

for i, sensor in enumerate(SENSOR_COLS):
    ax = axes[i]
    sample_units = train['unit_number'].unique()[:15]
    for uid in sample_units:
        unit_df = train[train['unit_number'] == uid]
        pct_life = unit_df['time_in_cycles'] / unit_df['time_in_cycles'].max()
        ax.plot(pct_life, unit_df[sensor], alpha=0.3, linewidth=0.8)
    ax.set_title(sensor, fontsize=9)
    ax.set_xlim(0, 1)

for j in range(len(SENSOR_COLS), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.suptitle('Sensor trajectories vs. % of engine life (15 sample units)', y=1.01)
plt.show()

Sensors with visually flat, noise-dominated trajectories across engine life
carry little RUL signal — these are dropped in `features.py` (`DROP_SENSORS`).
Sensors with a clear monotonic trend as the engine approaches end-of-life
are kept, and are exactly the ones that dominate feature importance in the
trained model (see `models/metadata.json` -> `top_features`).

## 3. Correlation with RUL (last-cycle snapshot)

To sanity check which raw sensors correlate most strongly with the RUL
label, independent of the rolling-feature engineering.

In [ ]:
corr = train[SENSOR_COLS + ['RUL']].corr()['RUL'].drop('RUL').sort_values()

plt.figure(figsize=(9, 6))
corr.plot(kind='barh', color=['#c4453a' if v < 0 else '#4fa8a0' for v in corr])
plt.title('Correlation of each raw sensor with RUL')
plt.xlabel('Pearson correlation')
plt.tight_layout()
plt.show()

## 4. RUL capping rationale

Early in an engine's life, RUL is not really predictable from sensor
readings alone (the engine looks 'healthy' for a long stretch regardless
of how much life is actually left) — so the label is capped
(see `RUL_CAP` in `features.py`) to avoid rewarding the model for
guessing arbitrarily large numbers on healthy engines, which published
CMAPSS work shows adds noise without adding real predictive value.

In [ ]:
plt.figure()
sample_unit = train[train['unit_number'] == train['unit_number'].unique()[0]]
plt.plot(sample_unit['time_in_cycles'], sample_unit['RUL'], color='#4fa8a0')
plt.xlabel('Cycle')
plt.ylabel('RUL (capped)')
plt.title('RUL label shape for one engine (piecewise: flat then linear decay)')
plt.show()

## Next steps

See `src/features.py` for the rolling-window feature pipeline and
`src/train.py` for model comparison (Linear Regression / Random Forest /
Gradient Boosting) using GroupKFold cross-validation and the official
NASA asymmetric scoring function.